In [1]:
import faiss
import numpy as np
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

c:\ProjectFiles\Research-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

dimension = 512 
index = faiss.IndexFlatL2(dimension)
image_paths = [] 

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 31558.18it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def add_to_index(path):
    img = Image.open(path)
    inputs = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs).cpu().numpy()
    
    faiss.normalize_L2(emb)
    index.add(emb)
    image_paths.append(path)


In [4]:
def coarse_search(query_path, k=5):
    query_img = Image.open(query_path)
    inputs = processor(images=query_img, return_tensors="pt")
    with torch.no_grad():
        q_emb = model.get_image_features(**inputs).cpu().numpy()
    
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, k)
    
    return [image_paths[i] for i in indices[0]]

In [5]:
from sentence_transformers import util

def refine_results(query_path, top_k_results, alpha=0.6):
    """
    query_path: Path to query image
    top_k_results: List of dicts with {'path': ..., 'visual_score': ...}
    alpha: Weight for visual similarity (1-alpha is for semantic)
    """
    query_img = Image.open(query_path)
    

    q_caption = vlm.caption(query_img, length="short")["caption"]
    q_env = vlm.query(query_img, "What is the setting of this image? (e.g. outdoors, shelf, handheld)")["answer"]
    query_context_text = f"{q_caption}. Setting: {q_env}"
    
    print(f"🔍 Query Context: {query_context_text}")
    
    refined_list = []
    
    for match in top_k_results:
        match_img = Image.open(match['path'])
        
        # 2. Extract Candidate Context
        m_caption = vlm.caption(match_img, length="short")["caption"]
        m_env = vlm.query(match_img, "What is the setting of this image?")["answer"]
        match_context_text = f"{m_caption}. Setting: {m_env}"
        

        q_text_emb = text_model.encode(query_context_text, convert_to_tensor=True)
        m_text_emb = text_model.encode(match_context_text, convert_to_tensor=True)
        semantic_score = util.cos_sim(q_text_emb, m_text_emb).item()
        

        final_score = (alpha * match['visual_score']) + ((1 - alpha) * semantic_score)
        
        refined_list.append({
            "path": match['path'],
            "original_rank": match['rank'],
            "visual_score": match['visual_score'],
            "semantic_score": semantic_score,
            "final_score": final_score,
            "reasoning": f"Q: {q_caption} | M: {m_caption}"
        })

    return sorted(refined_list, key=lambda x: x['final_score'], reverse=True)

In [ ]:
import matplotlib.pyplot as plt

# 1. Setup Test Data (Replace with actual paths on your machine)
# Tip: Use one "Distractor" (visually similar but semantically different)
test_gallery = [
    r"img\apple1.jpg",
    r"img\apple2.jpg",
    r"img\apple3.jpg",
    r"img\apple4.jpg"
]
query_image = r"img\query_apple.jpg"

# 2. Index the gallery
print("--- Indexing Gallery ---")
# Clear previous index if testing multiple times
index.reset() 
image_paths.clear()
for path in test_gallery:
    add_to_index(path)

# 3. Run Stage 1: Coarse Search
print("\n--- Running Stage 1: Coarse Search ---")
# Modified search logic to include scores for the refiner
inputs = processor(images=Image.open(query_image), return_tensors="pt")
with torch.no_grad():
    q_emb = model.get_image_features(**inputs).cpu().numpy()
faiss.normalize_L2(q_emb)
distances, indices = index.search(q_emb, k=len(test_gallery))

# Format for Stage 2
coarse_matches = []
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    # Convert L2 distance to a 0-1 similarity score
    vis_score = 1 / (1 + dist) 
    coarse_matches.append({
        'path': image_paths[idx],
        'visual_score': vis_score,
        'rank': rank + 1
    })

# 4. Run Stage 2: Refinement
print("--- Running Stage 2: Semantic Refinement ---")
# alpha=0.4 gives more weight to the VLM logic
refined_results = refine_results(query_image, coarse_matches, alpha=0.4)

# 5. Display Results Comparison
print("\n" + "="*50)
print(f"{'Path':<30} | {'Old Rank':<8} | {'New Rank':<8} | {'Final Score'}")
print("-" * 50)

for i, res in enumerate(refined_results):
    new_rank = i + 1
    path_tail = res['path'].split('/')[-1] # Just the filename for clarity
    print(f"{path_tail:<30} | {res['original_rank']:<8} | {new_rank:<8} | {res['final_score']:.4f}")
    print(f"   Reasoning: {res['reasoning']}")

# Optional: Visualize Top Match
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Query")
plt.imshow(Image.open(query_image))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Top Refined Match (Score: {refined_results[0]['final_score']:.2f})")
plt.imshow(Image.open(refined_results[0]['path']))
plt.axis('off')
plt.show()

--- Indexing Gallery ---


OSError: [Errno 22] Invalid argument: 'C:\\ProjectFiles\\Research-\\img\x07pple1.jpg'